# HySpecNet Mamba Qualitative Reconstruction

This notebook loads a small set of HySpecNet `DATA.npy` example patches from Google Drive, runs the best HySpecNet Mamba checkpoint, and visualizes:

1. original RGB patch,
2. reconstructed RGB patch,
3. absolute reconstruction error map,
4. original vs reconstructed spectral curve for a selected pixel.

Default checkpoint: `hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt`.


## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
EXAMPLE_ROOT = DRIVE_HSI / 'data/hyspecnet_examples/mamba_qualitative_examples'
CHECKPOINT_PATH = DRIVE_HSI / 'checkpoints/hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_0003_best.pt'
OUTPUT_DIR = DRIVE_HSI / 'analysis/hyspecnet_mamba_qualitative'

RGB_BANDS = (150, 100, 50)
PIXEL_MODE = 'max_error'  # one of: 'max_error', 'center', 'manual'
MANUAL_PIXEL = (64, 64)   # used only when PIXEL_MODE == 'manual'
USE_BITSTREAM = True      # True: model.compress/decompress; False: forward pass
USE_AMP = True
REQUIRE_CUDA = True
FORCE_REINSTALL_ENV = False

print('Example root:', EXAMPLE_ROOT)
print('Checkpoint:', CHECKPOINT_PATH)
print('Output dir:', OUTPUT_DIR)


## 2. Mount Drive and Prepare Repo

In [ ]:
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

print('Repo HEAD:')
subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)


## 3. Install Dependencies

This cell reuses the working Mamba Colab setup from `hyperview2_mamba_finetune_colab.ipynb`: Torch 2.7.1 with CUDA 12.6 plus prebuilt `causal-conv1d` and `mamba-ssm` wheels. On the first run it intentionally restarts the runtime after installing binary modules. After reconnect, rerun from the repo cell; the marker file makes this cell skip reinstalling.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, '-m', 'pip']
ENV_MARKER = Path('/content/.hsi_compression_hyspecnet_qual_env_v1_torch27_mamba232')


def run(cmd, *, required=True):
    print('Running:', ' '.join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        message = f'Command failed with exit code {result.returncode}: {" ".join(map(str, cmd))}'
        if required:
            raise RuntimeError(message)
        print('Optional command failed:', message)
        return False
    return True


if FORCE_REINSTALL_ENV and ENV_MARKER.exists():
    ENV_MARKER.unlink()

if not ENV_MARKER.exists():
    run(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools<82', 'wheel', 'packaging', 'pybind11', 'ninja'])
    run(PIP + [
        'install', '-q', '--force-reinstall',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    run(PIP + [
        'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'scipy>=1.12,<1.15', 'scikit-learn>=1.6,<1.8',
    ])
    run(PIP + ['install', '-q', '-e', '.[downstream]', 'eotdl', 'tqdm', 'matplotlib', 'ipywidgets'])

    import torch
    cxx11_abi = 'TRUE' if getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', True) else 'FALSE'
    python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    if python_tag != 'cp312':
        raise RuntimeError(f'This prebuilt Mamba preset expects Python 3.12, got {python_tag}.')
    if cxx11_abi != 'TRUE':
        raise RuntimeError(f'This prebuilt Mamba preset expects Torch CXX11 ABI TRUE, got {cxx11_abi}.')
    print('Torch CXX11 ABI:', cxx11_abi)

    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    ENV_MARKER.write_text('installed\n', encoding='utf-8')
    print('Dependencies installed. Restarting runtime to reload binary modules.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency marker exists, skipping reinstall:', ENV_MARKER)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab choose Runtime -> Change runtime type -> GPU.')
if not torch.__version__.startswith('2.7.'):
    if ENV_MARKER.exists():
        ENV_MARKER.unlink()
    raise RuntimeError(
        f'Mamba prebuilt wheels require Torch 2.7.x, but active Torch is {torch.__version__}. '
        'Restart the Colab runtime and rerun from the repo cell.'
    )
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    subprocess.run(['nvidia-smi'], check=False)

try:
    from mamba_ssm import Mamba  # noqa: F401
    print('mamba-ssm import: ok')
except Exception as exc:
    raise RuntimeError(
        'mamba-ssm is required for this notebook. Use a fresh GPU Colab runtime, '
        'set FORCE_REINSTALL_ENV=True, and rerun from the repo cell.'
    ) from exc


## 4. Load Example Patch Manifest

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

manifest_path = EXAMPLE_ROOT / 'manifest.json'
if not manifest_path.exists():
    raise FileNotFoundError(f'Missing example manifest: {manifest_path}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')

manifest = json.loads(manifest_path.read_text())
samples = manifest['samples']
print('Examples:', len(samples))
display(pd.DataFrame(samples)[['id', 'file', 'shape', 'min', 'max', 'mean']])


## 5. Build and Load the Mamba Model

In [ ]:
import math
from collections import OrderedDict

import torch

from hsi_compression.models.registry import build_model

ENTROPY_RUNTIME_KEYS = (
    'entropy_bottleneck._offset',
    'entropy_bottleneck._quantized_cdf',
    'entropy_bottleneck._cdf_length',
)


def load_checkpoint_model(checkpoint_path: Path, in_channels: int, device: torch.device):
    raw = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    cfg = raw.get('config', {})
    model_section = cfg.get('model', {})
    model_name = model_section.get('model_name')
    if not model_name:
        raise ValueError('Checkpoint does not contain config.model.model_name')
    kwargs = dict(model_section.get('model_kwargs', {}))
    kwargs.pop('in_channels', None)

    model = build_model(model_name=model_name, in_channels=in_channels, **kwargs).to(device)
    state = raw['model_state_dict']
    filtered = OrderedDict(
        (key, value)
        for key, value in state.items()
        if not any(key == runtime_key for runtime_key in ENTROPY_RUNTIME_KEYS)
    )
    missing, unexpected = model.load_state_dict(filtered, strict=False)
    unexpected = list(unexpected)
    not_runtime_missing = [key for key in missing if not any(key == r for r in ENTROPY_RUNTIME_KEYS)]
    if unexpected or not_runtime_missing:
        raise RuntimeError(f'Unexpected load_state_dict result: missing={missing}, unexpected={unexpected}')
    if hasattr(model, 'update'):
        model.update(force=True)
    model.eval()
    return model, cfg, raw

first_arr = np.load(EXAMPLE_ROOT / samples[0]['file'])
in_channels = int(first_arr.shape[0])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, cfg, raw_checkpoint = load_checkpoint_model(CHECKPOINT_PATH, in_channels, device)

print('Device:', device)
print('Model:', cfg.get('model', {}).get('model_name'))
print('Experiment:', cfg.get('experiment', {}).get('name'))
print('Checkpoint epoch:', raw_checkpoint.get('epoch'))
print('Best val loss:', raw_checkpoint.get('best_val_loss'))


## 6. Reconstruction and Plot Helpers

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch


def cube_to_rgb(cube: np.ndarray, bands=RGB_BANDS, params=None):
    channels = []
    if params is None:
        params = []
        for band in bands:
            plane = cube[band]
            lo, hi = np.percentile(plane, [2.0, 98.0])
            if hi <= lo:
                hi = lo + 1e-8
            params.append((float(lo), float(hi)))
    for band, (lo, hi) in zip(bands, params):
        channels.append(np.clip((cube[band] - lo) / (hi - lo), 0.0, 1.0))
    return np.stack(channels, axis=-1), params


def choose_pixel(x: np.ndarray, x_hat: np.ndarray):
    if PIXEL_MODE == 'manual':
        row, col = MANUAL_PIXEL
        return int(row), int(col)
    if PIXEL_MODE == 'center':
        return x.shape[1] // 2, x.shape[2] // 2
    err = np.mean(np.abs(x_hat - x), axis=0)
    return tuple(int(v) for v in np.unravel_index(np.argmax(err), err.shape))


def metric_summary(x: np.ndarray, x_hat: np.ndarray):
    diff = x_hat - x
    mse = float(np.mean(diff ** 2))
    mae = float(np.mean(np.abs(diff)))
    psnr = float(10.0 * math.log10(1.0 / max(mse, 1e-12)))
    x_flat = x.reshape(x.shape[0], -1).T
    y_flat = x_hat.reshape(x_hat.shape[0], -1).T
    denom = np.linalg.norm(x_flat, axis=1) * np.linalg.norm(y_flat, axis=1)
    valid = denom > 1e-12
    cos = np.ones(x_flat.shape[0], dtype=np.float64)
    cos[valid] = np.sum(x_flat[valid] * y_flat[valid], axis=1) / denom[valid]
    sam = float(np.degrees(np.mean(np.arccos(np.clip(cos, -1.0, 1.0)))))
    return {'mse': mse, 'mae': mae, 'psnr_db': psnr, 'sam_deg': sam}


@torch.no_grad()
def reconstruct_cube(cube: np.ndarray):
    x = torch.from_numpy(cube).unsqueeze(0).float().to(device)
    start = time.perf_counter()
    used = 'bitstream' if USE_BITSTREAM else 'forward'
    try:
        with torch.autocast(device_type=device.type, enabled=USE_AMP and device.type == 'cuda', dtype=torch.float16):
            if USE_BITSTREAM:
                packed = model.compress(x)
                decoded = model.decompress(**packed)
                x_hat = decoded['x_hat']
            else:
                x_hat = model(x)['x_hat']
    except Exception as exc:
        print('Bitstream/forward path failed, retrying plain forward. Error:', repr(exc))
        used = 'forward_fallback'
        with torch.autocast(device_type=device.type, enabled=False):
            x_hat = model(x)['x_hat']
    elapsed = time.perf_counter() - start
    x_hat = x_hat[0].detach().cpu().float().clamp(0.0, 1.0).numpy()
    return x_hat, used, elapsed


def plot_panel(sample: dict, x: np.ndarray, x_hat: np.ndarray, metrics: dict, output_path: Path):
    row, col = choose_pixel(x, x_hat)
    rgb_orig, rgb_params = cube_to_rgb(x)
    rgb_recon, _ = cube_to_rgb(x_hat, params=rgb_params)
    err = np.mean(np.abs(x_hat - x), axis=0)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), gridspec_kw={'width_ratios': [1, 1, 1, 1.45]})
    axes[0].imshow(rgb_orig)
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(rgb_recon)
    axes[1].set_title('Reconstruction')
    axes[1].axis('off')

    im = axes[2].imshow(err, cmap='magma')
    axes[2].set_title('Mean abs error')
    axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    band_axis = np.arange(x.shape[0])
    axes[3].plot(band_axis, x[:, row, col], color='black', linewidth=2.0, label='original')
    axes[3].plot(band_axis, x_hat[:, row, col], color='#d55e00', linewidth=1.6, label='reconstruction')
    axes[3].set_title(f'Spectrum at pixel ({row}, {col})')
    axes[3].set_xlabel('Band')
    axes[3].set_ylabel('Normalized reflectance')
    axes[3].grid(alpha=0.25)
    axes[3].legend()

    fig.suptitle(
        f"{sample['id']} | PSNR={metrics['psnr_db']:.3f} dB | SAM={metrics['sam_deg']:.3f} deg | MAE={metrics['mae']:.6f}",
        fontsize=11,
    )
    fig.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    display(fig)
    plt.close(fig)


## 7. Run Reconstruction for All Example Patches

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for index, sample in enumerate(samples):
    path = EXAMPLE_ROOT / sample['file']
    print(f"[{index + 1}/{len(samples)}] Loading {path.name}")
    x = np.load(path).astype(np.float32)
    if x.shape[0] != in_channels:
        raise ValueError(f"Unexpected channel count for {path}: {x.shape}")
    x_hat, mode_used, elapsed = reconstruct_cube(x)
    metrics = metric_summary(x, x_hat)
    metrics.update({'sample_id': sample['id'], 'mode_used': mode_used, 'time_sec': elapsed})
    rows.append(metrics)
    out_png = OUTPUT_DIR / f"{index:02d}_{sample['id']}_mamba_panel.png"
    plot_panel(sample, x, x_hat, metrics, out_png)
    np.savez_compressed(OUTPUT_DIR / f"{index:02d}_{sample['id']}_reconstruction.npz", original=x, reconstruction=x_hat)
    print('Saved:', out_png)

summary = pd.DataFrame(rows)
summary_path = OUTPUT_DIR / 'summary.csv'
summary.to_csv(summary_path, index=False)
print('Saved summary:', summary_path)
display(summary)


## 8. Optional Interactive Spectrum Browser

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    recon_files = sorted(OUTPUT_DIR.glob('*_reconstruction.npz'))
    if not recon_files:
        raise FileNotFoundError('No reconstruction files found. Run the previous cell first.')

    dropdown = widgets.Dropdown(
        options=[(path.name, str(path)) for path in recon_files],
        description='sample',
        layout=widgets.Layout(width='900px'),
    )
    row_slider = widgets.IntSlider(value=64, min=0, max=127, step=1, description='row')
    col_slider = widgets.IntSlider(value=64, min=0, max=127, step=1, description='col')

    def show(path_str, row, col):
        clear_output(wait=True)
        display(widgets.VBox([dropdown, row_slider, col_slider]))
        data = np.load(path_str)
        x = data['original']
        x_hat = data['reconstruction']
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(x[:, row, col], color='black', linewidth=2.0, label='original')
        ax.plot(x_hat[:, row, col], color='#d55e00', linewidth=1.6, label='reconstruction')
        ax.set_title(f'Spectrum at pixel ({row}, {col})')
        ax.set_xlabel('Band')
        ax.set_ylabel('Normalized reflectance')
        ax.grid(alpha=0.25)
        ax.legend()
        plt.show()

    ui = widgets.interactive_output(show, {'path_str': dropdown, 'row': row_slider, 'col': col_slider})
    display(widgets.VBox([dropdown, row_slider, col_slider]), ui)
except Exception as exc:
    print('Interactive browser unavailable:', repr(exc))
